<div align="center">
  <img src="https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/wsu_logo_horizontal.png" alt="Wayne State University Logo" width="320">
  <h1>Hyperparameter Tuning and Regularization</h1>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/practice/week12/practice12_hyperparameter_tuning_and_regularization.ipynb)

[Course Repository](https://github.com/WSU-AI-in-ME/ai-in-me-1) / [Practice Index](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/Practice_index.ipynb)

### ME 5995 - AI in Mechanical Engineering I: Fundamentals of Manufacturing Data Science
**Wayne State University / Optional, ungraded self-study / Normally not submitted**  
**Version: Student starter**

Try your selected exercises before consulting the solution. Each answer is one
possible approach. [Open the self-check solution](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/practice/week12/practice12_hyperparameter_tuning_and_regularization_solution.ipynb).

## Student Information

Optional for your personal copy. Double-click this Markdown cell to edit.

**Student name:** TODO: Enter your name  
**WSU AccessID:** TODO: Enter your WSU AccessID

## Learning objectives and routes

Estimated time: 65 minutes. Part A (25 minutes) uses a small network to inspect weight updates, decision boundaries and dropout masks. Part B (40 minutes) uses a manufacturing-sound dataset to select a model by validation cross-entropy and explain its errors.

Run the supplied code in order. Before each experiment, predict the effect of the change. Then edit the indicated value and compare the evidence. Part B can also be run independently from its setup cell. For local work, install NumPy and matplotlib in your Python environment for Part A. Part B also requires pandas and TensorFlow.

## Part A. Move a decision boundary by changing one setting

**25 minutes.** Use the eight synthetic points and a two-class network with four fixed ReLU hidden units. Each point has two dimensionless features. The four fixed hidden units return the positive parts of $x_1$, $-x_1$, $x_2$, and $-x_2$. Class A has score zero. Class B has score $w_1 h_1 + w_2 h_2 + w_3 h_3 + w_4 h_4 - 0.30$. The decision boundary is where those scores tie.

Use NumPy and matplotlib for this part. Run the supplied setup once. Each experiment starts from the same four weights, [0.24, -0.80, 1.00, -0.50]. Edit one value in an experiment cell and rerun just that cell. The dashed line shows the starting boundary, the solid line shows the new boundary, and the numbers above each plot give the four updated weights. Circles represent class A and triangles represent class B.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

toy_points = np.array([[1.7, -0.6], [1.3, 0.3], [1.0, 1.4], [0.6, -1.25],
                       [-1.0, 0.7], [-1.4, -0.6], [-0.8, 1.6], [-0.4, -1.5]])
toy_labels = np.array([1, 1, 1, 1, 0, 0, 0, 0])
toy_start = np.array([0.24, -0.80, 1.00, -0.50])
toy_order = np.array([0, 4, 5, 7, 1, 2, 3, 6])

def toy_hidden(points):
    x1, x2 = points.T
    return np.maximum(0, np.column_stack([x1, -x1, x2, -x2]))

def toy_fit(rate, batch, steps=1, penalty=None, strength=0.0):
    if not isinstance(batch, (int, np.integer)) or not 1 <= batch <= 8:
        raise ValueError("Choose an integer batch size from 1 through 8.")
    if not np.isfinite(rate) or not np.isfinite(strength) or rate < 0 or strength < 0:
        raise ValueError("Choose finite, nonnegative learning rates and penalty strengths.")
    indices = toy_order[:batch]
    h = toy_hidden(toy_points[indices])
    w = toy_start.copy()
    for step in range(steps):
        score = h @ w - 0.30
        probability = np.exp(-np.logaddexp(0, -score))
        gradient = h.T @ (probability - toy_labels[indices]) / batch
        w = w - rate * gradient
        # Apply the proximal penalty step after the data update.
        if penalty == "L1":
            w = np.sign(w) * np.maximum(np.abs(w) - rate * strength, 0)
        elif penalty == "L2":
            w = w / (1 + 2 * rate * strength)
    return w

### Supplied plotting code

Run this cell once. The plots share axes and colors so a change of scale cannot mimic a change in the boundary.

In [ ]:
def toy_compare(setting, values):
    fig, axes = plt.subplots(1, len(values), figsize=(4 * len(values), 3.8),
                             squeeze=False, sharex=True, sharey=True)
    axis_values = np.linspace(-2, 2, 225)
    gx, gy = np.meshgrid(axis_values, axis_values)
    grid_h = toy_hidden(np.column_stack([gx.ravel(), gy.ravel()]))
    start_scores = (grid_h @ toy_start - 0.30).reshape(gx.shape)
    for ax, value in zip(axes[0], values):
        batch = value if setting == "Batch size" else 8
        rate = value if setting == "Learning rate" else 0.8
        if setting in ("L1", "L2"):
            w = toy_fit(rate, batch, steps=8, penalty=setting, strength=value)
        else:
            w = toy_fit(rate, batch)
        scores = (grid_h @ w - 0.30).reshape(gx.shape)
        ax.contourf(gx, gy, scores > 0, levels=[-0.5, 0.5, 1.5],
                    colors=["#DDEFEA", "#FBE9DE"])
        if scores.min() < 0 < scores.max():
            ax.contour(gx, gy, scores, levels=[0], colors=["#18343C"], linewidths=2.5)
        else:
            ax.text(0.02, 0.03, "One predicted class in view", transform=ax.transAxes)
        ax.contour(gx, gy, start_scores, levels=[0], colors=["#707D83"], linestyles="--")
        for label, marker, color in [(0, "o", "#00675C"), (1, "^", "#BF5928")]:
            points = toy_points[toy_labels == label]
            ax.scatter(*points.T, marker=marker, color=color, edgecolor="white", zorder=4)
        if setting == "Batch size":
            chosen = toy_points[toy_order[:batch]]
            ax.scatter(*chosen.T, s=135, facecolors="none", edgecolors="black", zorder=5)
        weights = ", ".join(f"{weight:.5g}" for weight in w)
        ax.set_title(f"{setting} = {value:g}\n[{weights}]", fontsize=10)
        ax.set(xlabel="Feature 1", aspect="equal", xlim=(-2, 2), ylim=(-2, 2))
    axes[0, 0].set_ylabel("Feature 2")
    fig.tight_layout()
    plt.show()

### Learning rate: how far does one step move?

Keep the full batch of eight points. Predict how the first weight changes when the rate doubles. Run the initial settings, then change the middle rate to 1.6 and rerun. Compare the change **from 0.24** and the location of the solid boundary.

In [ ]:
# TODO: Try the change described above and rerun this cell.
toy_rates = [0.1, 0.8, 2.0]  # Edit the middle value and rerun.
toy_compare("Learning rate", toy_rates)

How does doubling the rate from 0.8 to 1.6 change the first weight update?

**Your answer:** TODO

### Batch size: which points supply the gradient?

Hold the rate at 0.8 and take one step. Rings identify the points used. Larger batches take more points from the same fixed order. Change the middle batch to 2 or 6. Does the update always shrink as the batch grows? Explain what the newly included points contribute.

In [ ]:
# TODO: Try the change described above and rerun this cell.
toy_batches = [1, 4, 8]  # Choose integers from 1 through 8.
toy_compare("Batch size", toy_batches)

Why does changing the batch change the boundary after just one update?

**Your answer:** TODO

### L1 and L2: which connections survive?

Use all eight points for eight updates at rate 0.8. After each data-gradient step, L1 subtracts a threshold from the magnitude of each weight and clips at zero. L2 divides each weight by a common factor greater than one for positive strength. These are **proximal updates**. The first panel in each row uses zero penalty.

Run both cells. Then set the middle strength to 0.10 in both and compare the third and fourth weights. If both become exactly zero, can feature 2 still affect the score? Check your answer against the orientation of the boundary.

In [ ]:
# TODO: Try the change described above and rerun this cell.
toy_l1_strengths = [0.0, 0.06, 0.18]
toy_compare("L1", toy_l1_strengths)

In [ ]:
# TODO: Try the change described above and rerun this cell.
toy_l2_strengths = [0.0, 0.06, 0.18]
toy_compare("L2", toy_l2_strengths)

At strength 0.18, compare the two connections from feature 2. How does that affect the boundary?

**Your answer:** TODO

### Dropout: which hidden units reach the output?

Use the input **(1, 1.4)**. Its four ReLU activations are **[1, 0, 1.4, 0]**. Keep the four initial output weights fixed and examine three forward passes. At training time each unit has probability $p$ of being masked. A kept activation is divided by $1-p$. At inference, all four original activations pass through unchanged.

Run the supplied function cell once. In the experiment cell below it, edit the rate and rerun. The default seed reproduces the two masks used in the worked example at $p=0.50$. Holding the seed fixed reuses the random draws as you change the rate. Changing the seed samples another pair of passes.

In [ ]:
from matplotlib.colors import ListedColormap

def show_toy_dropout(rate, seed):
    if not 0 <= rate < 1:
        raise ValueError("Choose a dropout rate from 0 up to, but excluding, 1.")
    h = toy_hidden(np.array([[1.0, 1.4]]))[0]
    rng = np.random.default_rng(seed)
    keep = rng.random((2, 4)) >= rate
    training = h * keep / (1 - rate)
    activations = np.vstack([training, h])
    scores = activations @ toy_start - 0.30
    prob_b = np.exp(-np.logaddexp(0, -scores))

    # Cell color records the mask state independently of the activation value.
    states = np.vstack([keep.astype(int), np.full(4, 2)])
    fig, (units, predictions) = plt.subplots(
        1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1.6, 1]},
        layout="constrained")
    units.imshow(states, cmap=ListedColormap(["#F5D6C9", "#CDE8DF", "#DCE7F5"]),
                 vmin=0, vmax=2, aspect="auto")
    units.set_xticks(range(4), ["h1 = 1", "h2 = 0", "h3 = 1.4", "h4 = 0"])
    units.set_yticks(range(3), ["Training A", "Training B", "Inference"])
    units.set_title("Hidden output after dropout")
    units.set_xlabel("Hidden activation before dropout")
    state_labels = ["Masked", "Kept", "Passed"]
    for row in range(3):
        for col in range(4):
            label = state_labels[states[row, col]]
            units.text(col, row, f"{label}\n{activations[row, col]:.3g}",
                       ha="center", va="center", color="#17323B")

    predictions.barh(range(3), prob_b, height=0.56, color=["#177365", "#177365", "#456B9A"])
    predictions.axvline(0.5, ls="--", color="#425354", linewidth=1)
    predictions.set(xlim=(0, 1), ylim=(2.5, -0.5), yticks=[],
                    xlabel="Probability of class B", title="Network prediction")
    for row, probability in enumerate(prob_b):
        winner = "B" if scores[row] > 0 else "A"
        predictions.text(0.02, row, f"{probability:.3f}  (class {winner})",
                         va="center", color="white", fontweight="bold")
    fig.suptitle(f"Input (1, 1.4)    |    Dropout rate {rate:g}    |    Seed {seed}")
    plt.show()

In [ ]:
# TODO: Compare rates 0.50, 0.00 and 0.80 with the same seed.
toy_dropout_rate = 0.50  # Try 0.00, 0.25 and 0.80.
toy_dropout_seed = 142  # Change this after comparing the rates.
show_toy_dropout(toy_dropout_rate, toy_dropout_seed)

**Read the figure.** At 0.50, which pass keeps h3? Trace that choice into the predicted probability of B. Find a cell labeled **Kept** whose value is zero and explain where that zero came from. At 0.00, predict all three rows before rerunning. At 0.80, check whether a training prediction changes class. Which row stays the same across rates and seeds?

**Your answer:** TODO

## Part B. Select a network using MIMII validation recordings

**40 minutes.** The toy boundary made an update visible in two dimensions. Each sound segment now has 1,952 inputs, and the network learns both hidden and output weights. Use validation cross-entropy to compare four combinations of width and learning rate, then test L2 and dropout at the selected setting. Keras applies the L2 penalty to the hidden kernel during Adam optimization. Its strength is specific to this model and update rule.

Use a Python 3 Colab CPU runtime or a local Jupyter environment with TensorFlow, NumPy, pandas and matplotlib installed. Run the remaining cells in order. Setup checks for the course files locally, then downloads approximately 54 MB if needed.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

# Supplied setup. Find a local course checkout or cache verified course files.
folder = Path("mimii_logmel800")
for parent in (Path.cwd(), *Path.cwd().parents):
    candidate = parent / "data/mimii_logmel800"
    if (candidate / "logmel.npz").is_file():
        folder = candidate
        break
folder.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/data/mimii_logmel800/"
checks = {"logmel.npz": "c6e1763b28155d7d08580176d53572953ed9f23475b5a199310087898daecd3a",
          "recordings.csv": "17f6261df25d44c5c99db761c7e8f632dd247281d9edbe391c2c6533a017341b"}
for name, expected in checks.items():
    path = folder / name
    if not path.exists():
        temp = path.with_suffix(".download")
        urlretrieve(base + name, temp)
        assert hashlib.sha256(temp.read_bytes()).hexdigest() == expected
        temp.replace(path)
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected
print("TensorFlow", tf.__version__)

## Fixed recording partitions

The course package contains 800 normal recordings. Each parent contributes ten one-second log-mel segments. Predict fan, pump, slider or valve from each segment. All siblings stay together. Machine IDs recur across partitions. The unit for each reported prediction is one segment. Ten segments from a recording are correlated observations.

In [ ]:
records = pd.read_csv(folder / "recordings.csv")
with np.load(folder / "logmel.npz", allow_pickle=False) as data:
    parent_row = data["recording_index"]
    assert np.array_equal(records["recording_id"].astype(str), data["recording_id"].astype(str))
    partition = records["partition"].to_numpy()[parent_row]
    train = partition == "train"
    valid = partition == "validation"
    X = data["X"]
    y = data["y"]
    X_train, y_train = X[train], y[train]
    X_valid, y_valid = X[valid], y[valid]
    del X
assert len(y_train) == 4800 and len(y_valid) == 1600
mean, sd = X_train.mean(), X_train.std()
X_train = (X_train - mean) / sd
X_valid = (X_valid - mean) / sd
print("Training and validation segment shapes:", X_train.shape, X_valid.shape)

## 1. Design the grid

Two hidden widths and two learning rates make four independent fits. For each pair, start a new model and optimizer. Hold the recording split, scaler, seed, batch size, epoch cap and selection rule fixed. Here grid search uses one predefined validation partition. A grouped three-fold search would require three fits per configuration.

Compare **validation predictive cross-entropy** (CE, nats per segment). Keras total loss also includes any weight penalty. Monitor CE so every candidate is judged by the same prediction task. When predicted probability assigned to the true class rises, its CE falls.

In [ ]:
widths = [16, 32]
learning_rates = [0.0003, 0.001]
epoch_cap = 12
print("Grid fits:", len(widths) * len(learning_rates))

### Supplied training function

Follow the four arguments into the model. Width changes the number of learned weights. Learning rate changes the optimizer step. L2 acts on hidden weights. Dropout acts on hidden activations during training. Each run restores the checkpoint with the lowest validation CE, with patience of three epochs.

In [ ]:
def fit_candidate(width, learning_rate, l2_strength=0.0, dropout_rate=0.0):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(9)
    model = tf.keras.Sequential([
        tf.keras.Input(shape=(32, 61)),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(width, activation="relu",
                              kernel_regularizer=tf.keras.regularizers.L2(l2_strength)),
        tf.keras.layers.Dropout(dropout_rate, seed=9),
        tf.keras.layers.Dense(4, activation="softmax")
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss="sparse_categorical_crossentropy",
                  metrics=[tf.keras.metrics.SparseCategoricalCrossentropy(name="ce"), "accuracy"])
    stop = tf.keras.callbacks.EarlyStopping(monitor="val_ce", mode="min",
                                            patience=3, restore_best_weights=True)
    fit = model.fit(X_train, y_train, validation_data=(X_valid, y_valid),
                    epochs=epoch_cap, batch_size=32, callbacks=[stop], verbose=2)
    return model, fit.history

## 2. Fit every pair and read the response surface

The inner loop tries every rate at one width. Read horizontally across the heatmap to isolate the rate effect at fixed width. Read vertically to isolate the width effect at fixed rate.

In [ ]:
grid_rows = []
best_ce = np.inf
for width in widths:
    for rate in learning_rates:
        print(f"Fitting width {width}, rate {rate}")
        model, history = fit_candidate(width, rate)
        epoch = int(np.argmin(history["val_ce"]))
        score = history["val_ce"][epoch]
        grid_rows.append([width, rate, epoch + 1, score, history["val_accuracy"][epoch]])
        if score < best_ce:
            best_ce = score
            best_width, best_rate = width, rate
            baseline, baseline_history = model, history

columns = ["Width", "Rate", "Best epoch", "Validation CE", "Accuracy"]
grid = pd.DataFrame(grid_rows, columns=columns)
print(grid.round(5).to_string(index=False))
print("Selected width and rate:", best_width, best_rate)

In [ ]:
surface = grid.pivot(index="Width", columns="Rate", values="Validation CE")
fig, ax = plt.subplots(figsize=(6, 3))
image = ax.imshow(surface, cmap="YlGnBu", aspect="auto")
ax.set_xticks(range(len(surface.columns)), surface.columns)
ax.set_yticks(range(len(surface.index)), surface.index)
ax.set(xlabel="Adam learning rate", ylabel="Hidden width (units)")
for row in range(len(surface.index)):
    for col in range(len(surface.columns)):
        value = surface.iloc[row, col]
        color = "white" if image.norm(value) > 0.55 else "black"
        ax.text(col, row, f"{value:.3f}", ha="center", va="center", color=color)
fig.colorbar(image, ax=ax, label="Validation CE (nats)")
fig.tight_layout()
plt.show()

**Pause.** Identify the lowest cell. Compare the effect of increasing width at each rate. If the winner lies on a boundary, which neighboring value would you examine in a future search? A bounded grid answers which tested combination scored best under this training budget.

**Your answer:** TODO

## 3. Test two regularizers at the selected width and rate

Reuse the winning grid model for the reference. Train two additional candidates, changing one mechanism at a time. L2 adds a weight penalty. Dropout randomly suppresses a fraction of hidden activations during training. Make a prediction about each before running. To investigate strength yourself, edit one value below and rerun this section and the cells that follow.

In [ ]:
l2_strength = 0.001
dropout_rate = 0.20
l2_model, l2_history = fit_candidate(best_width, best_rate, l2_strength=l2_strength)
drop_model, drop_history = fit_candidate(best_width, best_rate, dropout_rate=dropout_rate)
names = ["Grid winner", "L2", "Dropout"]
models = [baseline, l2_model, drop_model]
histories = [baseline_history, l2_history, drop_history]
rows = []
for name, model, history in zip(names, models, histories):
    epoch = int(np.argmin(history["val_ce"]))
    scores = model.evaluate(X_valid, y_valid, verbose=0, return_dict=True)
    rows.append([name, len(history["ce"]), epoch + 1, scores["ce"], scores["accuracy"]])
comparison = pd.DataFrame(rows, columns=["Candidate", "Epochs run", "Best epoch",
                                         "Validation CE", "Accuracy"])
print(comparison.round(5).to_string(index=False))
winner = models[int(comparison["Validation CE"].argmin())]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for axis, name, history in zip(axes, names, histories):
    epochs = np.arange(1, len(history["ce"]) + 1)
    best = int(np.argmin(history["val_ce"]))
    axis.plot(epochs, history["ce"], label="Training")
    axis.plot(epochs, history["val_ce"], label="Validation")
    axis.scatter(best + 1, history["val_ce"][best], color="black", label="Restored")
    axis.set(xlabel="Epoch", title=name)
axes[0].set_ylabel("Segment CE (nats)")
axes[0].legend()
fig.tight_layout()
plt.show()

**Check.** Which change improved validation CE relative to the grid winner? Did accuracy rank the candidates the same way? Training CE for dropout is measured with masks active and changing weights. Validation CE uses inference mode. A curve gap alone cannot establish a causal explanation.

**Your answer:** TODO

## Look inside the trained MLP

Both candidates below have the same hidden width. The first plot uses all entries in the hidden-layer weight matrix. Compare the distributions with and without the L2 penalty, then check the measured validation CE before deciding which model to keep.

The second figure follows one validation segment through the dropout candidate. ReLU can already produce zeros before masking. A new zero at a previously positive position identifies a masked activation. Repeated training calls draw new masks. The validation prediction uses inference mode.

In [ ]:
base_weights = baseline.layers[1].kernel.numpy().ravel()
l2_weights = l2_model.layers[1].kernel.numpy().ravel()
bins = np.histogram_bin_edges(np.concatenate([base_weights, l2_weights]), bins=45)
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(base_weights, bins=bins, density=True, histtype="step", linewidth=2, label="Without L2")
ax.hist(l2_weights, bins=bins, density=True, histtype="step", linewidth=2, label="With L2")
ax.set(xlabel="Hidden-layer weight", ylabel="Density")
ax.legend()
fig.tight_layout()
plt.show()
print("Hidden-kernel L2 norm without penalty:", round(float(np.linalg.norm(base_weights)), 2))
print("Hidden-kernel L2 norm with penalty:", round(float(np.linalg.norm(l2_weights)), 2))

In [ ]:
flat_segment = drop_model.layers[0](X_valid[:1])
hidden = drop_model.layers[1](flat_segment)
drop_layer = drop_model.layers[2]
pass_a = drop_layer(hidden, training=True).numpy()[0]
pass_b = drop_layer(hidden, training=True).numpy()[0]
inference = drop_layer(hidden, training=False).numpy()[0]
values = np.vstack([hidden.numpy()[0], pass_a, pass_b, inference])
fig, ax = plt.subplots(figsize=(10, 3))
image = ax.imshow(values, cmap="viridis", aspect="auto", interpolation="nearest")
ax.set_yticks(range(4), ["Before dropout", "Training pass A", "Training pass B", "Inference"])
ax.set_xlabel("Hidden unit index (zero-based)")
fig.colorbar(image, ax=ax, label="Activation")
fig.tight_layout()
plt.show()
np.testing.assert_allclose(inference, hidden.numpy()[0])

## Which machine types account for the remaining errors?

Read across a row to see what the selected candidate predicted for one true class. Row recall is the fraction of that class recognized correctly. Compare that fraction with overall accuracy.

In [ ]:
predicted = winner.predict(X_valid, verbose=0).argmax(axis=1)
names = ["Fan", "Pump", "Slider", "Valve"]
counts = tf.math.confusion_matrix(y_valid, predicted, num_classes=4).numpy()
errors = pd.DataFrame(counts, index=names, columns=names)
errors.index.name = "True class"
errors.columns.name = "Predicted class"
errors["Recall"] = counts.diagonal() / counts.sum(axis=1)
print(errors.to_string())

## Interpretation and self-check

1. After comparing all three candidates, which would you retain under the stated selection rule? Cite its validation CE and restored epoch. Check whether accuracy ranks the models in the same order.
2. Identify the largest off-diagonal count, including the direction of the error. Before changing class weights, check class support and whether errors cluster within a few recordings.
3. Would either result support deployment to a previously unseen machine ID? Design a grouped evaluation to answer that question.

**Your answer:** TODO

## Optional challenge

Repeat the comparison at another dropout rate while keeping the selected width and learning rate fixed. Report the change in validation CE and whether the same class confusion dominates. Your result will depend on the training run.

## Completion checklist

Rendered checkboxes are not clickable. Edit `[ ]` to `[x]` in this Markdown cell.

- [ ] I compared the toy weight updates and boundaries.
- [ ] I traced two dropout masks into a predicted probability.
- [ ] I interpreted the validation grid and compared the regularizers.
- [ ] I identified a directed class confusion and the scope of the split.

## Sources

Harsh Purohit and colleagues, [MIMII public 1.0](https://zenodo.org/records/3384388), CC BY-SA 4.0. The [course feature package](https://github.com/WSU-AI-in-ME/ai-in-me-1/tree/main/data/mimii_logmel800) selects normal recordings and supplies channel-1 log-mel segments, machine-type labels and recording partitions. The original benchmark concerns anomalous sound detection.

[Keras EarlyStopping](https://keras.io/api/callbacks/early_stopping/) and [weight regularizers](https://keras.io/api/layers/regularizers/).